# 01 — Profiling Dataset NER (Google Colab)

Notebook pertama untuk proyek UTS **Article Named-Entity Recognition Service**. Jalankan sel dari atas ke bawah.

## Tujuan tahap ini
- Membaca ZIP yang berisi CSV dengan delimiter titik koma (`;`) dan menangani tanda kutip CSV.
- Memeriksa struktur kolom, jumlah baris, nilai kosong, bahasa, panjang artikel, dan duplikasi.
- Menghasilkan file audit untuk membantu pembersihan dan anotasi pada tahap berikutnya.

**Penting:** jangan mengunggah dataset mentah ke GitHub karena soal meminta agar salinan dataset tidak dikomit. Simpan data di sesi Colab/Drive kelompok.


## Pembagian kerja 3 anggota

1. **Anggota 1 — Data & anotasi:** profiling, audit duplikasi, pedoman label, dan koordinasi anotasi bersama.
2. **Anggota 2 — Model & evaluasi:** baseline NER, training/adaptasi, precision/recall/F1, dan analisis kesalahan.
3. **Anggota 3 — Aplikasi:** API, antarmuka web, validasi input, pengujian integrasi, dan README.

Ketiganya tetap melakukan pemeriksaan silang anotasi. Test set tidak boleh dipakai untuk tuning model.


In [ ]:
import io
import os
import json
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
SEED = 42
OUTPUT_DIR = Path('/content/ner_profiling_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Environment siap.')


## 1. Unggah dataset

Unggah ZIP dari dosen yang berisi CSV, atau unggah beberapa file CSV langsung. Dataset tetap berada di runtime Colab dan tidak dimasukkan ke repository.


In [ ]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise ValueError('Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih ZIP/CSV dataset.')
print('File diterima:', list(uploaded.keys()))


## 2. Baca semua CSV dengan aman

CSV pada soal menggunakan pemisah titik koma. `quotechar='"'` menjaga koma/titik koma di dalam teks artikel yang berada dalam tanda kutip agar tidak memecah kolom.


In [ ]:
def read_csv_bytes(data: bytes, source_name: str) -> pd.DataFrame:
    """Baca satu CSV tanpa mengubah isi kolom, lalu catat nama file sumber."""
    last_error = None
    for encoding in ('utf-8-sig', 'utf-8', 'latin-1'):
        try:
            frame = pd.read_csv(
                io.BytesIO(data), sep=';', quotechar='"', dtype=str,
                encoding=encoding, keep_default_na=True, on_bad_lines='warn'
            )
            frame.columns = [str(c).strip().lower() for c in frame.columns]
            frame['source_file_loaded'] = source_name
            return frame
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f'Gagal membaca {source_name}: {last_error}')

frames = []
for uploaded_name, blob in uploaded.items():
    lower_name = uploaded_name.lower()
    if lower_name.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(blob)) as zf:
            csv_names = [n for n in zf.namelist() if n.lower().endswith('.csv') and not n.startswith('__MACOSX/')]
            print(f'{uploaded_name}: ditemukan {len(csv_names)} CSV')
            for member in csv_names:
                frames.append(read_csv_bytes(zf.read(member), f'{uploaded_name}:{member}'))
    elif lower_name.endswith('.csv'):
        frames.append(read_csv_bytes(blob, uploaded_name))
    else:
        print(f'Dilewati (bukan ZIP/CSV): {uploaded_name}')

if not frames:
    raise ValueError('Tidak ditemukan file CSV. Unggah ZIP yang berisi CSV atau file .csv.')

raw_df = pd.concat(frames, ignore_index=True, sort=False)
print(f'Total baris mentah: {len(raw_df):,}')
print(f'Jumlah kolom: {len(raw_df.columns)}')
display(pd.DataFrame({'column': raw_df.columns, 'dtype': [str(t) for t in raw_df.dtypes]}))
display(raw_df.head(3))


## 3. Buat salinan analisis yang dinormalisasi

Sumber asli tidak ditimpa. Untuk analisis, sel teks di-trim dan satu apostrof pembuka dihapus jika ada (terlihat pada ekspor dataset yang digunakan kelompok). Bandingkan contoh sebelum/sesudah; jika apostrof memang bagian dari teks, jangan gunakan pembersihan ini untuk kolom tersebut.


In [ ]:
def normalize_export_value(value):
    if not isinstance(value, str):
        return value
    value = value.strip()
    # Menangani penanda apostrof ekspor di awal sel; raw_df tetap dipertahankan.
    if value.startswith("'"):
        value = value[1:].strip()
    return value

df = raw_df.copy()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].map(normalize_export_value)

print('Contoh title mentah dan ternormalisasi:')
title_col = 'title' if 'title' in raw_df.columns else None
if title_col:
    display(pd.DataFrame({'raw_title': raw_df[title_col].head(5), 'normalized_title': df[title_col].head(5)}))
else:
    print('Kolom title tidak ditemukan; periksa nama kolom di atas.')


## 4. Ringkasan struktur dan kelengkapan

Angka pada bagian ini dihitung dari data yang diunggah. Catat hasil aktual ini untuk laporan UTS.


In [ ]:
expected_columns = ['title', 'body', 'language', 'entities', 'original_id', 'source_name', 'date_published']
missing_columns = [c for c in expected_columns if c not in df.columns]
print('Baris mentah:', f'{len(raw_df):,}')
print('Baris setelah normalisasi teks:', f'{len(df):,}')
print('Kolom:', list(df.columns))
print('Kolom wajib yang tidak ditemukan:', missing_columns if missing_columns else 'Tidak ada')

profile_rows = []
for col in df.columns:
    profile_rows.append({
        'column': col,
        'dtype': str(df[col].dtype),
        'missing_or_blank': int(df[col].isna().sum() + df[col].astype('string').str.strip().eq('').fillna(False).sum()),
        'unique_non_null': int(df[col].nunique(dropna=True))
    })
profile_columns = pd.DataFrame(profile_rows)
display(profile_columns)
profile_columns.to_csv(OUTPUT_DIR / 'column_profile.csv', index=False)


## 5. Periksa judul, isi, bahasa, dan kolom entities


In [ ]:
def blank_count(series):
    return int(series.isna().sum() + series.astype('string').str.strip().eq('').fillna(False).sum())

for col in ['title', 'body', 'language', 'entities', 'original_id']:
    if col in df.columns:
        print(f'{col}: kosong = {blank_count(df[col]):,} dari {len(df):,}')

if 'entities' in df.columns:
    nonempty_entities = df.loc[df['entities'].notna() & df['entities'].astype('string').str.strip().ne(''), 'entities']
    print('\nContoh entities tidak kosong (maksimal 10):')
    display(nonempty_entities.head(10).to_frame())
    if nonempty_entities.empty:
        print('Semua entities kosong. Kolom ini tidak dapat dianggap sebagai anotasi acuan; anotasi span-level perlu dibuat dan ditinjau manusia.')


## 6. Distribusi bahasa


In [ ]:
if 'language' in df.columns:
    language_counts = df['language'].fillna('(kosong)').replace('', '(kosong)').value_counts().rename_axis('language').reset_index(name='rows')
    display(language_counts)
    language_counts.to_csv(OUTPUT_DIR / 'language_distribution.csv', index=False)
    plt.figure(figsize=(9, 4))
    plt.bar(language_counts['language'].astype(str), language_counts['rows'])
    plt.title('Distribusi Bahasa Artikel (baris sebelum deduplikasi)')
    plt.xlabel('Kode bahasa'); plt.ylabel('Jumlah baris'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
else:
    language_counts = pd.DataFrame(columns=['language', 'rows'])
    print('Kolom language tidak tersedia.')


## 7. Panjang artikel dan artikel yang perlu diperiksa


In [ ]:
if 'title' in df.columns:
    df['title_char_count'] = df['title'].fillna('').astype(str).str.len()
if 'body' in df.columns:
    df['body_char_count'] = df['body'].fillna('').astype(str).str.len()
    print('Statistik panjang body (karakter):')
    display(df['body_char_count'].describe(percentiles=[.25, .5, .75, .9, .95]).to_frame())
    plt.figure(figsize=(8, 4))
    plt.hist(df.loc[df['body_char_count'] <= df['body_char_count'].quantile(.99), 'body_char_count'], bins=30)
    plt.title('Distribusi Panjang Body (hingga persentil ke-99)')
    plt.xlabel('Karakter'); plt.ylabel('Jumlah artikel'); plt.tight_layout(); plt.show()
    suspect_short = df[df['body_char_count'] < 200].copy()
    print('Artikel dengan body < 200 karakter:', len(suspect_short))
    cols = [c for c in ['original_id', 'title', 'body', 'language', 'source_file_loaded', 'body_char_count'] if c in suspect_short.columns]
    display(suspect_short[cols].head(20))
    suspect_short.to_csv(OUTPUT_DIR / 'articles_to_review_short_body.csv', index=False)


## 8. Audit duplikasi

Dibedakan antara baris yang identik dan ID artikel yang muncul berkali-kali. Jika artikel dengan ID sama memiliki judul/isi/bahasa berbeda, tinjau sebelum menghapus.


In [ ]:
content_cols = [c for c in ['original_id', 'title', 'body', 'language'] if c in df.columns]
exact_duplicate_mask = df.duplicated(subset=[c for c in df.columns if c != 'source_file_loaded'], keep='first')
print('Baris duplikat identik (tanpa kolom sumber file):', int(exact_duplicate_mask.sum()))
exact_duplicates = df.loc[exact_duplicate_mask].copy()
exact_duplicates.to_csv(OUTPUT_DIR / 'exact_duplicate_rows.csv', index=False)

if 'original_id' in df.columns:
    valid_id = df['original_id'].notna() & df['original_id'].astype('string').str.strip().ne('')
    duplicate_id_mask = valid_id & df.duplicated(subset=['original_id'], keep=False)
    ids_repeated = df.loc[duplicate_id_mask].copy()
    repeat_counts = (df.loc[valid_id].groupby('original_id', dropna=False).size().rename('row_count').reset_index())
    repeated_id_counts = repeat_counts[repeat_counts['row_count'] > 1].sort_values('row_count', ascending=False)
    print('ID artikel unik:', int(df.loc[valid_id, 'original_id'].nunique()))
    print('Jumlah ID yang muncul berulang:', len(repeated_id_counts))
    print('Baris di dalam kelompok ID berulang:', int(ids_repeated.shape[0]))
    repeated_id_counts.to_csv(OUTPUT_DIR / 'repeated_id_counts.csv', index=False)
    ids_repeated.to_csv(OUTPUT_DIR / 'repeated_id_audit.csv', index=False)
    if content_cols:
        conflict_table = df.loc[valid_id].groupby('original_id')[content_cols[1:]].nunique(dropna=False)
        conflict_ids = conflict_table[(conflict_table > 1).any(axis=1)].reset_index()
        print('ID yang memiliki konflik title/body/language:', len(conflict_ids))
        display(conflict_ids.head(20))
        conflict_ids.to_csv(OUTPUT_DIR / 'conflicting_article_ids.csv', index=False)
else:
    print('Tidak ada original_id. Deduplikasi artikel perlu memakai kombinasi kolom dan aturan yang ditinjau manusia.')


## 9. Membuat dataset awal setelah deduplikasi (untuk ditinjau)

Tahap ini hanya menyiapkan data awal. Jangan gunakan sebagai training sebelum konflik ID diperiksa dan anotasi entitas dibuat. Jika ada konflik ID, notebook memberi peringatan dan hanya membuang duplikat yang benar-benar identik pada kolom isi.


In [ ]:
if 'original_id' in df.columns:
    conflict_path = OUTPUT_DIR / 'conflicting_article_ids.csv'
    conflict_count = len(pd.read_csv(conflict_path)) if conflict_path.exists() and conflict_path.stat().st_size > 1 else 0
    dedupe_fields = [c for c in ['original_id', 'title', 'body', 'language'] if c in df.columns]
    if conflict_count == 0:
        articles = df.drop_duplicates(subset=['original_id'], keep='first').copy()
        dedupe_note = 'Satu baris per original_id; tidak ditemukan konflik metadata berdasarkan pemeriksaan kolom yang tersedia.'
    else:
        # Jangan memilih salah satu versi yang konflik secara diam-diam.
        exact_fields = [c for c in df.columns if c != 'source_file_loaded']
        articles = df.drop_duplicates(subset=exact_fields, keep='first').copy()
        dedupe_note = f'Ditemukan {conflict_count} ID konflik; hanya duplikat identik dihapus. Tinjau conflicting_article_ids.csv sebelum training.'
else:
    exact_fields = [c for c in df.columns if c != 'source_file_loaded']
    articles = df.drop_duplicates(subset=exact_fields, keep='first').copy()
    dedupe_note = 'original_id tidak tersedia; hanya baris identik yang dihapus.'

articles.to_csv(OUTPUT_DIR / 'articles_normalized_for_review.csv', index=False)
print(dedupe_note)
print('Jumlah baris untuk review lanjutan:', f'{len(articles):,}')
display(articles.head(5))


## 10. Ekspor hasil profiling

ZIP berisi CSV audit, distribusi bahasa, profil kolom, dan dataset awal untuk ditinjau. File ini tidak berisi label anotasi buatan; anotasi dibuat pada tahap berikutnya.


In [ ]:
summary = {
    'raw_rows': int(len(raw_df)),
    'rows_after_text_normalization': int(len(df)),
    'column_count': int(len(df.columns)),
    'columns': list(df.columns),
    'missing_required_columns': missing_columns,
    'language_distribution': language_counts.to_dict(orient='records') if 'language_counts' in globals() else [],
    'entities_nonempty': int((df['entities'].notna() & df['entities'].astype('string').str.strip().ne('')).sum()) if 'entities' in df.columns else None,
    'normalized_rows_for_review': int(len(articles)),
    'deduplication_note': dedupe_note,
    'random_seed': SEED,
    'note': 'Entities kosong/tidak span-labeled tidak boleh dianggap sebagai gold-standard tanpa validasi manusia.'
}
with open(OUTPUT_DIR / 'profiling_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

import shutil
archive_path = shutil.make_archive('/content/NER_Profiling_Results', 'zip', OUTPUT_DIR)
print('Selesai. Arsip hasil:', archive_path)
display(Markdown('### Cek sebelum lanjut\n- [ ] Simpan ZIP hasil profiling.\n- [ ] Baca `conflicting_article_ids.csv` (jika ada).\n- [ ] Tentukan bahasa utama untuk model.\n- [ ] Jangan mulai training sebelum data anotasi span-level ditinjau manusia.'))
files.download(archive_path)
